<a href="https://colab.research.google.com/github/YOUR_GITHUB_USERNAME/YOUR_REPO_NAME/blob/main/agent_orchestrator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 AI Agent Orchestrator: From Naive to Production-Ready

### ⚠️ The Problem: "The Authority Trap"
In most AI agent tutorials, the LLM is given direct access to tools. If the model hallucinates or decides to perform an action, it happens instantly.
* **Risky:** No human-in-the-loop for high-stakes actions.
* **Brittle:** Hard to handle API errors, retries, or rate limits.
* **Unpredictable:** The model is given **Authority** when it should only have **Reasoning**.

### ✅ The Solution: The Orchestrator Pattern
This notebook demonstrates how to separate **Intent** from **Execution**. 

1. **The Model** generates a "Function Call" (The Intent).
2. **The System** (your code) intercepts that intent.
3. **The Logic Layer** validates the intent against business rules (The Guardrails).
4. **The Tool** only executes if the logic layer gives the green light.

---

### 🛠️ Architecture Flow
**User Input** → **LLM** → **Structured Intent (JSON)** → 🛡️ **ORCHESTRATOR (Safety Check)** → **Tool Execution** → **Final Result**

---

### 🚀 How to use this Boilerplate:
1. **Define Tools:** See how we wrap "real-world" actions in safe functions.
2. **The Orchestrator:** Observe the "Guardrails" (like spending limits and balance checks).
3. **Test Scenarios:** Run the simulations to see how the system rejects unsafe LLM decisions before they cause damage.

In [1]:
import json

In [2]:
# 1. Define the Tools (The Execution Layer)
def get_user_balance(user_id):
    # In production, this hits your DB
    db = {"user_123": 1000.00}
    return db.get(user_id, 0)

def trigger_wire_transfer(amount, recipient):
    # This is a 'Side Effect'
    print(f"💰 SUCCESS: ${amount} sent to {recipient}")
    return True

In [3]:
# 2. The Logic Layer (The "Manager")
def orchestrator(llm_response):
    """
    Acts as the boundary between reasoning and action.
    """
    intent = llm_response.get("tool_calls")[0]
    func_name = intent["function"]
    args = intent["arguments"]

    print(f"🔍 Orchestrator: LLM wants to call '{func_name}' with {args}")

    # --- SAFETY BOUNDARIES ---
    if func_name == "trigger_wire_transfer":
        if args["amount"] > 500:
            return "ERROR: Transfer exceeds limit. Requires human approval."
        
        # Check balance before executing
        balance = get_user_balance("user_123")
        if args["amount"] > balance:
            return "ERROR: Insufficient funds."

    # --- EXECUTION ---
    if func_name == "trigger_wire_transfer":
        result = trigger_wire_transfer(args["amount"], args["recipient"])
        return f"Action complete: {result}"

In [4]:
# 3. Simulated "Naive" LLM Output
mock_llm_intent = {
    "tool_calls": [{
        "function": "trigger_wire_transfer",
        "arguments": {"amount": 250, "recipient": "External_Account_ABC"}
    }]
}

In [5]:
# Run the flow
status = orchestrator(mock_llm_intent)
print(f"🏁 Final Result: {status}")

🔍 Orchestrator: LLM wants to call 'trigger_wire_transfer' with {'amount': 250, 'recipient': 'External_Account_ABC'}
💰 SUCCESS: $250 sent to External_Account_ABC
🏁 Final Result: Action complete: True
